# Offline Mode (Self-Managed vLLM) — Speculative Decoding Training

This notebook demonstrates how to train a custom **Eagle3 draft model** for speculative
decoding using the `OFFLINE` mode of `SpeculativeDecodingTrainer` from the Kubeflow SDK
on Red Hat OpenShift AI.

## What is OFFLINE Mode?

The `OFFLINE` mode connects to a **self-managed external vLLM server** to extract hidden
states from the verifier model, then trains the Eagle3 draft model — all within a single
job. Unlike `ONLINE` mode, the SDK does **not** deploy a vLLM sidecar. You provide a
`vllm_endpoint` pointing to your own vLLM instance.

This is useful when you already have a vLLM deployment running (e.g., as an OpenShift AI
model serving instance) and want to reuse it for hidden state extraction.

## How It Works

1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC
3. Training runs immediately after extraction completes — all within the same job

## Speculative Decoding Overview

Large language models generate tokens one at a time, and each token requires reading the
entire model from GPU memory — making inference **memory-bound**. Speculative decoding
exploits this: a small, fast **draft model** (~1.2 GB with Qwen3-0.6B) guesses the next several tokens,
then the large **verifier model** checks all guesses in a single forward pass. The output
is mathematically identical to normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate
layers of the verifier (not just the final logits), giving it richer context for more
accurate predictions.

## Dataset

This example uses the `magpie` built-in dataset (Magpie-format conversation dataset).

## Hardware Requirements

The table below shows the **minimum** resources needed. See the Configuration cell for
recommended values that improve training speed.

| Component | GPU (min) | GPU (recommended) | CPU (min) | CPU (rec.) | Memory (min) | Memory (rec.) |
|-----------|-----------|-------------------|-----------|------------|-------------|--------------|
| Training container | 1× GPU | 2× GPU | 1 core | 4 cores | 32Gi | 64Gi |
| External vLLM server | 1× GPU | 1× GPU | 1 core | 4 cores | 48Gi | 96Gi |

> The external vLLM server is self-managed — its resources are separate from the TrainJob.

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth
from kubernetes import client as k8s

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

The following environment variables are required for API authentication:

- `OPENSHIFT_API_URL` — your cluster API URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` — an access token for API calls

In OpenShift AI workbenches, these are often auto-set.

If they are not set in your environment, uncomment and populate the values in the next cell.

In [ ]:
# ============================================================================
# AUTHENTICATION
# ============================================================================
# If your workbench does not auto-populate these env vars, uncomment and fill them in:
#
# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set. "
        "Either set them in your environment or uncomment the values above."
    )

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC MOUNT PATHS
# ============================================================================
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name/mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT AND CLUSTER TRAINING RUNTIME
# ============================================================================
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(client_configuration=configuration)
)

# ClusterTrainingRuntime (CTR) for OFFLINE mode.
# OFFLINE mode does not use a managed vLLM sidecar, so only the model optimization CTR is needed.
MODEL_OPT_CTR = "speculator-model-opt-cuda"  # Training only, no vLLM sidecar

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found" if MODEL_OPT_CTR in available_runtimes else "WARNING: not found on cluster"
)
print(f"CTR '{MODEL_OPT_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## (Optional) Download the Verifier Model

OFFLINE mode requires the verifier model on the shared PVC (as a PVC URI). If the
model is not already on your PVC, download it here. The external vLLM server must
also have access to this same model on the shared PVC.

Skip this cell if the model is already on your PVC.

In [ ]:
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = HF_TOKEN

model_id = "Qwen/Qwen3-0.6B"
local_dir = f"{NOTEBOOK_SHARED_PATH}/models/Qwen3-0.6B"

snapshot_download(model_id, local_dir=local_dir)
print(f"Model downloaded to {local_dir}")

## Configuration

The following constants configure the training run. The verifier model is
[Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), a 28-layer transformer,
specified as a PVC URI since OFFLINE mode requires the model on shared storage.

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

You must also set `VLLM_ENDPOINT` to point to your external vLLM server serving
the same verifier model.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
# OFFLINE mode with an external vLLM endpoint requires a PVC URI — the external
# vLLM server already has the model loaded from shared storage, so the training
# pod reads the model config from the same PVC path.
VERIFIER_MODEL_PVC_URI = f"pvc://{PVC_NAME}/models/Qwen3-0.6B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-0.6B has 28 transformer layers (indexed 1-28).
# Layers chosen: early (2), mid (14), late (25), and final (28) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
# When using a PVC URI for verifier_model, target_layer_ids MUST be set explicitly
# because the SDK cannot auto-detect them from the PVC.
TARGET_LAYER_IDS = [2, 14, 25, 28]

# Minimum resources for the training container.
# 1 GPU is sufficient to train the small Eagle3 draft model (~1.2 GB with Qwen3-0.6B).
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 1,  # Recommended: 2 — enables data-parallel training
    "cpu": "1",  # Recommended: "4" — faster data loading and preprocessing
    "memory": "32Gi",  # Recommended: "64Gi" — more headroom for optimizer state
}

# URL of your externally managed vLLM server.
# This must be a running vLLM instance serving the same verifier model (Qwen3-0.6B).
# The /v1 path exposes the OpenAI-compatible API that the SDK calls for extraction.
VLLM_ENDPOINT = "http://vllm-svc.speculative-decoding.svc.cluster.local:8000/v1"

# Training hyperparameters
EPOCHS = 3  # Number of full passes over the training data
LEARNING_RATE = 1e-4  # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for both extraction and training
MAX_SAMPLES = 500  # Cap on the number of dataset samples to process

print("OFFLINE Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL_PVC_URI}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  vLLM endpoint:     {VLLM_ENDPOINT}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## Offline Mode (Self-Managed vLLM)

The `OFFLINE` mode extracts hidden states via a self-managed external vLLM server,
then trains the draft model in a single job. This is useful when you already have a
vLLM deployment running (e.g., as an OpenShift AI model serving instance) and want to
reuse it for hidden state extraction instead of having the SDK deploy a sidecar.

**How it works:**
1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC at `hidden_states_path`
3. Training runs immediately after extraction completes — all within the same job

**Key differences from other modes:**
- You must provide `vllm_endpoint` pointing to your external vLLM server
- The SDK does not deploy a vLLM sidecar — `vllm_resources` is not used
- Both `training_resources` (for the training container) and `vllm_endpoint`
  (for extraction) are required
- The external vLLM server must be in the **same namespace** and have access to the
  **same shared PVC** as the TrainJob

We use the `magpie` built-in dataset for this example.

In [ ]:
OFFLINE_JOB = f"eagle3-offline-{RUN_NAME}"
OFFLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-offline"

# Configure the OFFLINE trainer.
# OFFLINE mode connects to an external vLLM endpoint for extraction, then trains.
# Both steps happen within the same job — extraction first, training second.
# Unlike DATA_ONLY + TRAIN_ONLY, this is a single-job workflow.
# Unlike ONLINE, the SDK does NOT deploy a vLLM sidecar — you manage it yourself.
offline_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.OFFLINE,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL_PVC_URI,
    dataset_name="magpie",  # Built-in Magpie conversation dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_endpoint=VLLM_ENDPOINT,  # External vLLM server URL
    hidden_states_path=f"{OFFLINE_OUTPUT}/hidden_states",  # Where extracted states are saved
    training_resources=TRAINING_RESOURCES,  # Resources for the training container
    regenerate_responses=True,  # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=OFFLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        resume_from_checkpoint=True,  # Resume from the latest checkpoint if one exists
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("OFFLINE Configuration:")
print(f"  Job name:         {OFFLINE_JOB}")
print(f"  Mode:             {offline_trainer.mode.value}")
print(f"  Verifier:         {offline_trainer.verifier_model}")
print(f"  vLLM endpoint:    {offline_trainer.vllm_endpoint}")
print(f"  Dataset:          {offline_trainer.dataset_name}")
print(f"  Target layers:    {offline_trainer.config.target_layer_ids}")
print(f"  Hidden states:    {offline_trainer.hidden_states_path}")
print(f"  Output dir:       {offline_trainer.output_dir}")

In [ ]:
# Submit the OFFLINE TrainJob to the cluster.
# Uses MODEL_OPT_CTR — no SDK-managed vLLM sidecar (the external endpoint handles extraction).
trainer_client.train(
    options=[Name(name=OFFLINE_JOB)],
    trainer=offline_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"OFFLINE job submitted: {OFFLINE_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={OFFLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the OFFLINE job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(OFFLINE_JOB)

## (Optional) Test the Trained Draft Model

Validate the trained checkpoint, convert it from `Eagle3DraftModel` (training format) to `Eagle3Speculator` (vLLM format), and run speculative decoding inference with 3 test prompts.

> **Note:** The first cell below installs `speculators==0.6.0` compatibility fixes for the workbench environment. The second cell requires a GPU and ~8 GiB RAM.

In [ ]:
# Install the speculators SDK (no-deps avoids pulling in conflicting packages)
!pip install speculators==0.6.0 --no-deps --quiet

import importlib.util
import os
import re
import subprocess
import sys

# ---------------------------------------------------------------------------
# 1. kernels.LayerRepository — default version when revision/version omitted
# ---------------------------------------------------------------------------
# The kernels library raises ValueError when neither revision nor version is
# supplied. Speculators calls LayerRepository without these args, so we set a
# safe default (version=1) both on disk and in the live Python process.

_kll = sys.modules.get("kernels.layer.layer")
if _kll is None:
    try:
        import kernels.layer.layer as _kll
    except ImportError:
        _kll = None

if _kll is not None:
    # On-disk fix: replace the hard ValueError with a safe default
    with open(_kll.__file__) as _f:
        _src = _f.read()
    _old = 'raise ValueError("Either a revision or a version must be specified.")'
    if _old in _src:
        with open(_kll.__file__, "w") as _f:
            _f.write(_src.replace(_old, "version = 1"))
        print(
            f"Installed: kernels LayerRepository default version fix — {_kll.__file__}"
        )
    else:
        print("Already installed: kernels LayerRepository default version fix")

    # In-memory fix: monkey-patch __init__ to inject version=1 at runtime
    _orig_init = _kll.LayerRepository.__init__

    def _patched_init(self, *args, **kwargs):
        if kwargs.get("revision") is None and kwargs.get("version") is None:
            kwargs["version"] = 1
        _orig_init(self, *args, **kwargs)

    _kll.LayerRepository.__init__ = _patched_init
else:
    print("Already installed: kernels LayerRepository fix (on-disk from previous run)")

# ---------------------------------------------------------------------------
# 2. speculators train/data.py — make 'datasets' import optional
# ---------------------------------------------------------------------------
# The speculators training module unconditionally imports 'datasets', which is
# not installed in the workbench environment. Wrapping it in try/except allows
# the converter (which doesn't need datasets) to load without errors.

_spec = importlib.util.find_spec("speculators")
if _spec:
    _train_data = os.path.join(os.path.dirname(_spec.origin), "train", "data.py")
    with open(_train_data) as _f:
        _src = _f.read()
    _old_import = "from datasets import load_from_disk\n"
    _new_import = (
        "try:\n"
        "    from datasets import load_from_disk\n"
        "except ImportError:\n"
        "    load_from_disk = None\n"
    )
    if _old_import in _src and _new_import not in _src:
        with open(_train_data, "w") as _f:
            _f.write(_src.replace(_old_import, _new_import))
        print(f"Installed: optional datasets import — {_train_data}")
    else:
        print("Already installed: optional datasets import")

# ---------------------------------------------------------------------------
# 3. Eagle3Converter — flatten nested transformer_layer_config
# ---------------------------------------------------------------------------
# Eagle3DraftModel stores hidden_size, intermediate_size, and num_heads inside
# a nested 'transformer_layer_config' dict. Eagle3Converter reads them from
# the top level and gets None. This fix promotes nested keys to top level so
# the converter can find the correct dimensions during checkpoint conversion.

if _spec:
    _converter = os.path.join(
        os.path.dirname(_spec.origin), "convert", "eagle", "eagle3_converter.py"
    )
    with open(_converter) as _f:
        _src = _f.read()
    # Inject a block that lifts transformer_layer_config keys to the top level
    _old_block = (
        "    ) -> LlamaConfig:\n"
        "        # Load target model config for vLLM compatibility\n"
        "        try:"
    )
    _new_block = (
        "    ) -> LlamaConfig:\n"
        '        if "transformer_layer_config" in eagle_config:\n'
        '            _tlc = eagle_config["transformer_layer_config"]\n'
        "            eagle_config = {**eagle_config}\n"
        "            for _k, _v in _tlc.items():\n"
        "                eagle_config.setdefault(_k, _v)\n"
        "\n"
        "        # Load target model config for vLLM compatibility\n"
        "        try:"
    )
    if _new_block not in _src and _old_block in _src:
        with open(_converter, "w") as _f:
            _f.write(_src.replace(_old_block, _new_block))
        print(f"Installed: Eagle3Converter config flattening — {_converter}")
    else:
        print("Already installed: Eagle3Converter config flattening")

# ---------------------------------------------------------------------------
# 4. vLLM flashinfer sampling — disable JIT compilation
# ---------------------------------------------------------------------------
# The RHOAI workbench container is missing curand.h, which causes flashinfer's
# JIT-compiled sampling kernels to fail at runtime. This replaces every
# 'from flashinfer.sampling import' statement in vLLM source files with a
# controlled ImportError, forcing vLLM to fall back to its built-in sampler.

_vllm_spec = importlib.util.find_spec("vllm")
if _vllm_spec:
    _vllm_root = os.path.dirname(_vllm_spec.origin)
    # Find all vLLM files that import flashinfer sampling
    _result = subprocess.run(
        ["grep", "-rl", "from flashinfer.sampling", _vllm_root],
        capture_output=True,
        text=True,
    )
    _flashinfer_files = [
        fp for fp in _result.stdout.strip().split("\n") if fp.endswith(".py")
    ]
    for _fp in _flashinfer_files:
        with open(_fp) as _f:
            _src = _f.read()
        if "PATCHED_FLASHINFER_SAMPLING" in _src:
            print(
                f"Already installed: flashinfer sampling"
                f" disable — {os.path.basename(_fp)}"
            )
            continue
        # Replace import with ImportError so vLLM uses its built-in sampler
        _new_src = _src.replace(
            "from flashinfer.sampling import",
            'raise ImportError("PATCHED_FLASHINFER_SAMPLING")'  # noqa: E501
            "  # from flashinfer.sampling import",
        )
        if _new_src != _src:
            with open(_fp, "w") as _f:
                _f.write(_new_src)
            print(f"Installed: flashinfer sampling disable — {os.path.basename(_fp)}")
else:
    print("Skipped: vllm not installed — flashinfer fix not needed")

# ---------------------------------------------------------------------------
# 5. vLLM Eagle3 — fix combine_hidden_states dimension mismatch
# ---------------------------------------------------------------------------
# vLLM concatenates ALL aux hidden states along the feature dimension
# (e.g. 4 layers * 1024 = 4096), but the fc layer expects input shaped as
# 3 * target_hidden_size = 3072. The "3" represents three input components
# (token embedding + previous hidden state + combined aux) — a different
# convention than raw concatenation. This fix truncates the input tensor to
# match fc's expected in_features, keeping the first N features that align
# with the trained fc weight dimensions.

if _vllm_spec:
    _eagle3_path = os.path.join(
        os.path.dirname(_vllm_spec.origin),
        "model_executor",
        "models",
        "llama_eagle3.py",
    )
    if os.path.exists(_eagle3_path):
        with open(_eagle3_path) as _f:
            _src = _f.read()

        _dirty = False

        # Step A: Restore fc_input_size block to correct state.
        # Earlier patch attempts may have corrupted indentation or changed
        # the * 3 multiplier. This regex matches the block regardless of its
        # current state and replaces it with the correctly indented original.
        _block_re = re.compile(
            r'(\s*)if hasattr\(self\.config, ["\']target_hidden_size["\']\):\s*\n'
            r"(?:\s*#[^\n]*\n)*"
            r"(?:\s*\w[^\n]*\n)*?"
            r"\s*fc_input_size = self\.config\.target_hidden_size \* \S+\s*\n"
            r"\s*else:\s*\n"
            r"(?:\s*#[^\n]*\n)*"
            r"\s*fc_input_size = self\.config\.hidden_size \* \S+\s*\n",
        )
        _block_match = _block_re.search(_src)
        if _block_match:
            _indent = _block_match.group(1)
            _good_block = (
                f'{_indent}if hasattr(self.config, "target_hidden_size"):\n'
                f"{_indent}    fc_input_size = self.config.target_hidden_size * 3\n"
                f"{_indent}else:\n"
                f"{_indent}    fc_input_size = self.config.hidden_size * 3\n"
            )
            if _block_match.group(0) != _good_block:
                _src = (
                    _src[: _block_match.start()]
                    + _good_block
                    + _src[_block_match.end() :]
                )
                _dirty = True

        # Step B: Inject truncation logic into combine_hidden_states.
        # Before passing hidden_states to fc, check if the last dimension
        # exceeds fc's in_features and truncate to match. This handles the
        # case where num_aux_layers > 3 without changing the fc weight shape.
        _old_fc_call = "return self.model.fc(hidden_states)"
        _new_fc_call = (
            "fc_in = self.model.fc.weight.shape[-1]\n"
            "        if hidden_states.shape[-1] > fc_in:\n"
            "            hidden_states = hidden_states[..., :fc_in]\n"
            "        return self.model.fc(hidden_states)"
        )
        if "fc_in = self.model.fc" not in _src and _old_fc_call in _src:
            _src = _src.replace(_old_fc_call, _new_fc_call)
            _dirty = True

        if _dirty:
            with open(_eagle3_path, "w") as _f:
                _f.write(_src)
            print(f"Installed: Eagle3 combine_hidden_states fix — {_eagle3_path}")
        elif "fc_in = self.model.fc" in _src:
            print("Already installed: Eagle3 combine_hidden_states fix")
        else:
            print(f"Warning: could not locate target pattern in {_eagle3_path}")

# ---------------------------------------------------------------------------
# Reload modified modules so subsequent imports pick up all changes.
# Note: kernels is a PyO3 (Rust) extension and cannot be safely reloaded.
# ---------------------------------------------------------------------------
for _mod in list(sys.modules):
    if _mod.startswith((
        "speculators",
        "transformers.modeling",
        "transformers.integrations",
    )):
        del sys.modules[_mod]

In [ ]:
import json
import logging
import os
import shutil

import safetensors.torch as safetensors_torch

# Suppress verbose vLLM startup logs — show only errors and fatal messages.
# VLLM_LOGGING_LEVEL must be set before importing vLLM internals.
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
# Belt-and-suspenders: tell vLLM not to use flashinfer sampler (in case the
# on-disk patch above didn't cover a dynamically loaded path).
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from speculators.convert.eagle.eagle3_converter import Eagle3Converter
from vllm import LLM, SamplingParams

# Reduce Python-level log noise from vLLM and torch compilation
logging.getLogger("vllm").setLevel(logging.WARNING)
for _name in ("torch._dynamo", "torch._inductor"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# ── Resolve the verifier model's local filesystem path ──
# In offline mode, VERIFIER_MODEL_PVC_URI is always a pvc:// URI pointing to
# the model on the shared PVC. Convert it to a local filesystem path.
verifier_model_local = VERIFIER_MODEL_PVC_URI.replace(
    f"pvc://{PVC_NAME}", NOTEBOOK_SHARED_PATH
)

if not os.path.isdir(verifier_model_local):
    raise FileNotFoundError(
        f"Verifier model not found at {verifier_model_local}. "
        "Run the model download cell first."
    )

# ── Locate the best training checkpoint ──
# Training saves checkpoints under {RUN_NAME}/checkpoints/ with a
# 'checkpoint_best' symlink pointing to the highest-scoring checkpoint.
checkpoint_dir = f"{NOTEBOOK_SHARED_PATH}/speculator/{RUN_NAME}-offline/checkpoints"
checkpoint_best = os.path.join(checkpoint_dir, "checkpoint_best")

if os.path.islink(checkpoint_best):
    # Resolve the symlink to get the actual checkpoint directory name
    target = os.readlink(checkpoint_best)
    checkpoint_path = os.path.join(checkpoint_dir, target)
elif os.path.isdir(checkpoint_best):
    checkpoint_path = checkpoint_best
else:
    raise FileNotFoundError(
        f"No checkpoint found at {checkpoint_best}. "
        "Verify training completed successfully."
    )

# ── Display checkpoint metadata ──
# Load the draft model config to show architecture details and training metrics.
config_path = os.path.join(checkpoint_path, "config.json")
with open(config_path) as f:
    draft_config = json.load(f)

model_file = os.path.join(checkpoint_path, "model.safetensors")
size_gb = os.path.getsize(model_file) / (1024**3)

print("Draft model checkpoint:")
print(f"  Path: {checkpoint_path}")
print(f"  Architecture: {draft_config.get('architectures', ['unknown'])[0]}")
print(f"  Speculators version: {draft_config.get('speculators_version')}")
print(f"  Model size: {size_gb:.2f} GB")
if "eagle_aux_hidden_state_layer_ids" in draft_config:
    print(f"  Aux layer IDs: {draft_config['eagle_aux_hidden_state_layer_ids']}")

# Show validation metrics from the best checkpoint (if available)
metrics_path = os.path.join(checkpoint_path, "val_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    # Metrics are keyed by epoch (loss_0_epoch, loss_1_epoch, etc.) —
    # prefer the latest epoch's values
    loss = (
        metrics.get("loss_2_epoch")
        or metrics.get("loss_1_epoch")
        or metrics.get("loss_0_epoch")
    )
    acc = (
        metrics.get("full_acc_2_epoch")
        or metrics.get("full_acc_1_epoch")
        or metrics.get("full_acc_0_epoch")
    )
    if loss is not None:
        print(f"  Validation loss: {loss:.4f}")
    if acc is not None:
        print(f"  Full accuracy: {acc:.4f}")

# ── Convert from Eagle3DraftModel (training) to Eagle3Speculator (vLLM) ──
# The training checkpoint uses speculators' internal format. Eagle3Converter
# transforms it into the format vLLM expects for speculative decoding.
converted_path = checkpoint_path + "_vllm"

# Always re-convert to avoid stale artifacts from previous runs
if os.path.exists(converted_path):
    shutil.rmtree(converted_path)
    print(f"\nRemoved stale converted model at {converted_path}")

print("\nConverting Eagle3DraftModel to Eagle3Speculator (vLLM format)...")
converter = Eagle3Converter()
converter.convert(
    input_path=checkpoint_path,  # Source: training checkpoint directory
    output_path=converted_path,  # Destination: vLLM-compatible model directory
    base_model=verifier_model_local,  # Verifier model (needed for config alignment)
    validate=True,  # Run post-conversion validation checks
    norm_before_residual=True,  # Match the training architecture's norm placement
    # Verifier layers used for aux hidden states
    eagle_aux_hidden_state_layer_ids=TARGET_LAYER_IDS,
)
print(f"Converted to {converted_path}")

# ── Verify fc weight compatibility between training and converted checkpoints ──
# Eagle3Converter may produce fc weights with different dimensions than what
# the training checkpoint used. If shapes differ, copy the training weight.

converted_safetensors = os.path.join(converted_path, "model.safetensors")
training_safetensors = os.path.join(checkpoint_path, "model.safetensors")

converted_weights = safetensors_torch.load_file(converted_safetensors)
training_weights = safetensors_torch.load_file(training_safetensors)

# Extract and display fc layer weight shapes for comparison
converted_fc = {k: tuple(v.shape) for k, v in converted_weights.items() if "fc" in k}
training_fc = {k: tuple(v.shape) for k, v in training_weights.items() if "fc" in k}
print(f"\n  Converted fc weights: {converted_fc}")
print(f"  Training fc weights:  {training_fc}")

# If shapes mismatch, overwrite the converted fc weight with the training one
fc_fixed = False
for t_key, t_val in training_weights.items():
    if "fc" not in t_key or "weight" not in t_key:
        continue
    # Check multiple key conventions (with/without 'model.' prefix)
    for c_key in [t_key, f"model.{t_key}", t_key.replace("model.", "")]:
        if c_key in converted_weights and converted_weights[c_key].shape != t_val.shape:
            print(
                f"  Fixing {c_key}: "
                f"{tuple(converted_weights[c_key].shape)}"
                f" -> {tuple(t_val.shape)}"
            )
            converted_weights[c_key] = t_val
            fc_fixed = True
            break

if fc_fixed:
    safetensors_torch.save_file(converted_weights, converted_safetensors)
    print("  Saved corrected fc weights")
else:
    print("  fc weights match — no correction needed")

# ── Load the verifier + draft model with vLLM speculative decoding ──
# The verifier model is the main LLM; the draft model (Eagle3 speculator)
# proposes candidate tokens that the verifier then accepts or rejects.
print("\nLoading verifier + Eagle3 draft model for speculative decoding...")
llm = LLM(
    model=verifier_model_local,
    speculative_config={
        # Use Eagle3 speculative decoding algorithm
        "method": "eagle3",
        # Path to the converted draft model
        "model": converted_path,
        # Number of tokens the draft model proposes per step
        "num_speculative_tokens": 5,
    },
    # Use up to 90% of available GPU memory
    gpu_memory_utilization=0.9,
    # Allow custom model code from the checkpoint
    trust_remote_code=True,
    # Maximum sequence length (must match training)
    max_model_len=TOTAL_SEQ_LEN,
)

# ── Run test prompts through speculative decoding ──
# Use greedy decoding (temperature=0) for deterministic, reproducible output.
sampling_params = SamplingParams(temperature=0.0, max_tokens=512)

test_prompts = [
    "Explain what speculative decoding is in two sentences.",
    "Write a Python function to compute the factorial of a number.",
    "What are the three laws of thermodynamics?",
]

print("\n" + "=" * 60)
print("SPECULATIVE DECODING INFERENCE TEST")
print("=" * 60)

outputs = llm.generate(test_prompts, sampling_params)

for i, output in enumerate(outputs, 1):
    response = output.outputs[0].text
    num_tokens = len(output.outputs[0].token_ids)
    # Show prompt (truncated), response (truncated), and token count
    print(f"\n[{i}] {output.prompt[:80]}")
    print(f"    {response[:500]}")
    print(f"    ({num_tokens} tokens)")

# ── Release GPU memory ──
del llm
print("\nvLLM engine released")

## Cleanup

Delete the TrainJob when you are done.


In [ ]:
# Delete the completed TrainJob and inference service to free cluster resources.
# Note: Deleting these does NOT delete the output data on the PVC —
# checkpoints remain available for future use.

# Delete training job
# trainer_client.delete_job(OFFLINE_JOB)
# print(f"TrainJob '{OFFLINE_JOB}' deleted")

# Delete inference service
# SERVICE_NAME = f"eagle3-draft-{RUN_NAME}"

# If your workbench does not auto-populate this env variable, uncomment and fill it in:
# NAMESPACE = "your-namespace"
# NAMESPACE = os.getenv("NAMESPACE", "default")

# Note: You must have permission to create InferenceServices in this namespace.
# If you get a 403 error, contact your cluster admin or use a namespace where you have appropriate permissions.

# for kind in ["inferenceservice", "servingruntime"]:
#     result = subprocess.run(
#         ["oc", "delete", kind, SERVICE_NAME, "-n", NAMESPACE, "--ignore-not-found"],
#         capture_output=True,
#         text=True,
#     )
#     if result.returncode == 0:
#         print(f"  Deleted {kind} '{SERVICE_NAME}'")

## Summary

This notebook demonstrated **OFFLINE** mode — extracting hidden states via an external
vLLM endpoint and training an Eagle3 draft model in a single job using the `magpie`
dataset.

OFFLINE mode is ideal when you already have a vLLM deployment running and want to
reuse it for hidden state extraction instead of having the SDK deploy a managed sidecar.

### Key Takeaways

- `vllm_endpoint` points to your external vLLM server — the SDK does not deploy a sidecar
- Both extraction and training happen in a single job
- The external vLLM server must be serving the same verifier model used in training
- All storage paths use **PVC URIs** (`pvc://<pvc-name>/<path>`)

### Next Steps

- Deploy the trained draft model with vLLM for speculative decoding inference
- Adjust `epochs`, `lr`, and `max_samples` to tune draft model quality
- Try [DATA_ONLY + TRAIN_ONLY](../data-only/) to separate extraction from training
  and iterate on hyperparameters without re-running extraction
- Try [ONLINE](../online/) mode for the fully managed alternative where the SDK
  deploys the vLLM sidecar automatically